Dear Franziska,
The ceilometers owned by MET Norway are part of the EUMETNET network E-PROFILE. Due to compatibility reasons within this network, we deliver the normalized, range-corrected signal (beta_raw) even though the firmware is also capable of calculating the attenuated backscatter (beta_att). However, E-PROFILE receives and processes all our ceilometer data, adding the calibration constant and  attenuated backscatter. You can find daily files from most ceilometers in Europe, including Andøya, here: https://data.ceda.ac.uk/badc/eprofile/data/daily_files
You will probably have to set up a user account in order to download. Bulk downloads, or download via script is also possible. See e.g. here: https://help.ceda.ac.uk/article/5191-downloading-multiple-files-with-wget

Best regards,
Anette L Borg 

 Citable as:  Met Norway; EUMETNET (2022): EUMETNET E-PROFILE: ceilometer cloud base height and aerosol profile data from Met Norway's Lufft CHM15k "Nimbus" instrument deployed at Andoya, Norway. NERC EDS Centre for Environmental Data Analysis, 2025/05/01. https://catalogue.ceda.ac.uk/uuid/d3b2c99cc281457a9cfd0ab2efa49cbd/ 

In [1]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)


abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]



if "mimi" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"

INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

# make figure directory to put in reflectivity processed data
# try:
#     new_dir = pathlib.Path(FIG_DIR)
#     new_dir.mkdir(parents=True, exist_ok=True)
# except OSError:
#     pass

# make data directory to put in ceilometer processed data
OUT_DIR = os.path.join(OUTPUT_DATA_DIR, 'ceilometer/E_PROFILE_corrected')
try:
    new_dir = pathlib.Path(OUT_DIR)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

mimi.uio.no


In [2]:
# supress warnings
import warnings
warnings.filterwarnings('ignore') # don't output warnings

# import packages
from imports import(glob, xr, np, urlparse, parse_qs, BeautifulSoup, requests, plt, cm, BoundaryNorm, LinearSegmentedColormap,fct, datetime, DateFormatter, pd)

# from netCDF4 import Dataset
# from urllib.parse import urlparse



In [3]:
# reload imports
%load_ext autoreload
%autoreload 2

In [4]:
current_date = datetime.datetime.utcnow().strftime('%Y-%m-%d')

In [5]:
# 2021/03/16 - 2021/03/31
# 2022/03/16 - 2022/04/07
# 2023/02/12 - 2023/03/14
date_ranges = {
    "2021": pd.date_range("2021-03-16", "2021-03-30").strftime("%Y%m%d").tolist(),
    "2022": pd.date_range("2022-03-17", "2022-04-07").strftime("%Y%m%d").tolist(),
    "2023a": pd.date_range("2023-02-12", "2023-02-22").strftime("%Y%m%d").tolist(),
    "2023b": ["20230314"],  # single date as a list
}

In [ ]:
for key, dates in date_ranges.items():
    for date in dates:
        year    = date[0:4]
        month   = date[4:6]
        day     = date[6:8]
        
        files = sorted(glob(os.path.join(INPUT_DATA_DIR, 'ceilometer', 'E_PROFILE',f'L2_0-20000-0-01010_A{date}*')))
        ds = xr.open_mfdataset(files)
        
        # Apply mask: keep only attenuated_backscatter values where quality_flag == 0
        beta_att_clean = ds['attenuated_backscatter_0'].where(ds['quality_flag'] == 0, )
        # Convert units of backscatter
        beta_att_clean = beta_att_clean*1e-6


        beta_att_clean.attrs['units'] = 'm^-1.sr^-1'
        beta_att_clean.attrs['long_name'] = 'Attenuated Backscatter at wavelength 0'
        beta_att_clean.attrs['history'] = f'{current_date}: filter out attenuated_backscatter_0 values where quality_flag == 0'

        # replace in dataset
        ds['attenuated_backscatter_0'] = beta_att_clean
        ds = ds.transpose('time', 'altitude', 'layer')
        
        path = os.path.join(OUTPUT_DATA_DIR, 'ceilometer', 'E_PROFILE_corrected', f"L2_0-20000-0-01010_A{date}.nc")
        print(f"writing file: ... {path}")
        ds.to_netcdf(path)

writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210316.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210317.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210318.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210319.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210320.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210321.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210322.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210323.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_PROFILE_corrected/L2_0-20000-0-01010_A20210324.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/E_

Update token
https://help.ceda.ac.uk/article/5100-archive-access-tokens

In [7]:
# token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICI4ZjhmaUpyaUtDY3hmaHhzdU5vazVEekdJdFZ4amhhTWNJa05ZX2U4MnhJIn0.eyJleHAiOjE3NDYyNzk4NjUsImlhdCI6MTc0NjAyMDY2NSwianRpIjoiNjFjNjFiNmEtMzJhNC00NGY0LWFiZjktZmYzMGJhMTk0YWZjIiwiaXNzIjoiaHR0cHM6Ly9hY2NvdW50cy5jZWRhLmFjLnVrL3JlYWxtcy9jZWRhIiwic3ViIjoiMDlmYjk0MTMtNjU4Ni00ZWIxLWI1NGEtYzRmYjJiNTY0MmJhIiwidHlwIjoiQmVhcmVyIiwiYXpwIjoic2VydmljZXMtcG9ydGFsLWNlZGEtYWMtdWsiLCJzZXNzaW9uX3N0YXRlIjoiYjZlMjYyYzEtNGQ3Ny00ZDEwLTkxZmQtYTMyOGI1YjY2OTdkIiwiYWNyIjoiMSIsInNjb3BlIjoiZW1haWwgb3BlbmlkIHByb2ZpbGUgZ3JvdXBfbWVtYmVyc2hpcCIsInNpZCI6ImI2ZTI2MmMxLTRkNzctNGQxMC05MWZkLWEzMjhiNWI2Njk3ZCIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJvcGVuaWQiOiJodHRwczovL2NlZGEuYWMudWsvb3BlbmlkL0ZyYW56aXNrYS5IZWxsbXV0aDEiLCJuYW1lIjoiRnJhbnppc2thIEhlbGxtdXRoIiwicHJlZmVycmVkX3VzZXJuYW1lIjoiZmhlbGxtdXRoMDAxIiwiZ2l2ZW5fbmFtZSI6IkZyYW56aXNrYSIsImZhbWlseV9uYW1lIjoiSGVsbG11dGgiLCJlbWFpbCI6ImZyYW56aXNrYS5oZWxsbXV0aEBnZW8udWlvLm5vIn0.CETExXkl7u2Cgy3kTXIaU5-PY1kwM_92k437xZAWZovus4h2ameKjDZKKU8pHG6DxhF8pjWOsA3feVMjOem_tKELZecHNugMSPJhWQG56VI5-aKfBtb2fOeFUcgJKxpCj_1CLxbzAQvHqgTfRVFnQJymua7Fa4RkEatCEc-bPG3aGPHaTqdVdascSIFzdaTTuy269dGBDnisG7xvKPV_Uqzh_mjsRtr5vAy9BCC1nbkr291AIFgBxivmjjaXVIGiuBy3gG9LN_HRhJw_vsKzFj5s0F-1RQYglg7vRbiQxUJcNA8wxfqWHoU8qI8PqseJmfUveSxXjl3VuOeKPtilQQ"

In [8]:
# # Set up the headers with the access token
# headers = {
#   "Authorization": f"Bearer {token}"
# }

In [9]:
# for key, dates in date_ranges.items():
#     for date in dates:
#         year    = date[0:4]
#         month   = date[4:6]
#         day     = date[6:8]
        
#         url = f"https://dap.ceda.ac.uk/badc/eprofile/data/daily_files/norway/andoya/met-norway-lufft-chm15k_A/{year}/L2_0-20000-0-01010_A{date}.nc"#?download=1"
        
        
#         filename = os.path.basename(urlparse(url).path)
#         path_to_save = os.path.join(OUTPUT_DATA_DIR, 'ceilometer/E_PROFILE', filename)
#         response = requests.get(url, headers=headers, stream=True)


#         # Save to local file
#         if response.status_code == 200:
#             with open(path_to_save, 'wb') as f:
#                 for chunk in response.iter_content(chunk_size=8192):
#                     f.write(chunk)
#                 # f.write(response.content)
#             # f.close()
#             print(f"✅ Downloaded and saved: {filename}")
#         else:
#             print(f"❌ Failed to download {filename}, status code: {response.status_code}")


In [10]:
# with open(path_to_save, 'rb') as f:
#     head = f.read(500).decode(errors='ignore')
#     print(head)

In [11]:
# for var_name, var_data in ds.data_vars.items():
#     print(f"Variable: {var_name}", f"  Dimensions: {var_data.dims}", f"  Shape: {var_data.shape}", f"  Data type: {var_data.dtype}", "  Attributes:")
#     for attr_key, attr_val in var_data.attrs.items():
#         print(f"    {attr_key}: {attr_val}")
#     print("-" * 40)

In [12]:
# bsc = ds['attenuated_backscatter_0']
# cbh = ds['cloud_base_height'].isel(layer = 0).where(ds['cloud_base_height'].isel(layer = 0) > -1.)


# # plot figure
# fig, axsm = plt.subplots(1,1, figsize=[7,5],)
# cmap = cm.lipari_r  # define the colormap
# # extract all colors from the colormap
# cmaplist = [cmap(i) for i in range(cmap.N)]
# # force the first color entry to be white
# cmaplist[0] = (255,255,255, 0.9)
# # create the new map
# cmap = LinearSegmentedColormap.from_list(
# 'Custom cmap', cmaplist, 10) # cmap.N
# # define the bins and normalize
# # bounds = np.linspace(0, 7e7, 11)
# # norm = BoundaryNorm(bounds, cmap.N)


# # ceilometer backscatter   
# (bsc.where(bsc>=0.)).plot.pcolormesh(ax = axsm, x='time', cmap = cmap, norm = norm, cbar_kwargs = {'spacing': 'proportional','label': 'Backscatter'})

# #cloud base 
# cbh.plot(ax = axsm, x = 'time', linestyle="",marker=".", color = 'grey', alpha=0.3, markersize=2, add_legend=True, label='Cloud Base', )


# axsm.set_ylim([0, 5000])
# # axsm.set_xlim([datetime.date(str(year), str(month), str(day)), datetime.date(str(year), str(month), str(day+1))])
# day_start   = (datetime.datetime.strptime(f"{year}{month}{day} 00:00:00", '%Y%m%d %H:%M:%S').strftime('%Y-%m-%d %H:%M:%S'))
# day_end     = (datetime.datetime.strptime(f"{year}{month}{day} 00:00:00", '%Y%m%d %H:%M:%S') + datetime.timedelta(hours=23, minutes=59)).strftime('%Y-%m-%d %H:%M:%S')
# xmax = (datetime.datetime.strptime(day_end, '%Y-%m-%d %H:%M:%S') + datetime.timedelta(minutes=1,)).strftime('%Y-%m-%d %H:%M:%S')
# # axsm.set_xlim([day_start, xmax])
# # Set title and labels for axes
# axsm.set(xlabel="Time [UTC]",
# ylabel="height [m]",
# title="{}-{}-{}".format(year, month, day),)
# axsm.xaxis.grid()

# # # Define the date format
# # date_form = DateFormatter("%H:00")
# # axsm.xaxis.set_major_formatter(date_form)

# plt.tight_layout();

In [13]:
# This can be used to get beta_raw from the MetNo thredds server
# data_vars=['beta_raw_hr','range_gate_hr','range_gate','laser_pulses','base','scaling','p_calc','beta_raw','cbh', ]
# drop_variables=['latitude',
#  'longitude',
#  'azimuth',
#  'zenith',
#  'altitude',
#  'wavelength',
#  'average_time',
#  'life_time',
#  'error_ext',
#  'state_laser',
#  'state_detector',
#  'state_optics',
#  'temp_int',
#  'temp_ext',
#  'temp_det',
#  'temp_lom',
#  'stddev',
#  'tcc',
#  'bcc',
#  'sci',
#  'mxd',
#  'nn1',
#  'nn2',
#  'nn3',
#  'pbl',
#  'pbs',
#  'cbe',
#  'cdp',
#  'cde',
#  'cho',
#  'vor',
#  'voe',]
# for key, dates in date_ranges.items():
#     for date in dates:
#         year    = date[0:4]
#         month   = date[4:6]
#         day     = date[6:8]
        
#         if date != "20230225":
#             continue
#         else:
#             # Base URL for the THREDDS catalog
#             catalog_url = f"https://thredds.met.no/thredds/catalog/remotesensingceilometer/{year}/{month}/{day}/andoya/raw/catalog.html"
            
#             # Get all available NetCDF file URLs for the day
#             file_urls = fct.get_file_urls(catalog_url,day)
            
#             new_file_urls = []
#             for i in range(len(file_urls)):
#                 # print(file_urls[i])
                
#                 # Parse query parameters
#                 query_params = parse_qs(urlparse(file_urls[i]).query)
#                 # Extract dataset value
#                 dataset_path = query_params.get("dataset", [""])[0]
                
#                 new_file_urls.append(f"https://thredds.met.no/thredds/dodsC/{dataset_path}")
#             new_file_urls = sorted(new_file_urls)
            
#             # Load and merge all files
#             # clean up time across all files before combining them: (preprocess=lambda x: x.sortby('time'))
#             combined_ds = xr.open_mfdataset(new_file_urls, combine='by_coords', decode_times = True, preprocess=fct.clean_time, data_vars = data_vars, drop_variables=drop_variables)
            
#             # # Save to file
#             # _path = os.path.join(OUT_DIR, f'anx_ceil_{date}.nc')
#             # print(f'writing file: ... {_path}')
#             # combined_ds.to_netcdf(path = _path)